# Exercise 6: Streaming Simulation (Final Challenge)

Trong bài thực hành này, chúng ta sẽ mô phỏng xử lý streaming bằng cách đọc dữ liệu theo batch từ file JSON.
Đây là thử thách cuối cùng giúp tổng hợp các kiến thức đã học về Spark DataFrame.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp, hour, round, count, max, countDistinct, sum

spark = SparkSession.builder \
    .appName("Exercise6") \
    .master("spark://spark-master:7077") \
    .getOrCreate()

print("SparkSession đã được tạo thành công.")

**Giải thích cú pháp:**

- `SparkSession.builder`: Tạo một builder để cấu hình SparkSession. Đây là điểm khởi đầu của mọi ứng dụng Spark.
- `.appName("Exercise6")`: Đặt tên cho ứng dụng Spark, giúp nhận diện trên Spark UI.
- `.master("spark://spark-master:7077")`: Kết nối đến Spark Master đang chạy tại địa chỉ `spark-master` cổng `7077`.
- `.getOrCreate()`: Lấy SparkSession hiện có hoặc tạo mới nếu chưa tồn tại.
- Các hàm `col`, `to_timestamp`, `hour`, `round`, `count`, `max`, `countDistinct`, `sum` được import từ `pyspark.sql.functions` để sử dụng trong các bước xử lý tiếp theo.

**Kết quả mong đợi:** In ra dòng chữ "SparkSession đã được tạo thành công." nếu kết nối đến Spark Master không có lỗi.

In [ ]:
events_df = spark.read.json("/home/jovyan/data/json/web_events.json")

events_df.show(5)

**Giải thích cú pháp:**

- `spark.read.json(...)`: Đọc file JSON từ đường dẫn được cung cấp và trả về một DataFrame. Spark sẽ tự động suy luận (infer) schema từ dữ liệu.
- `.show(5)`: Hiển thị 5 dòng đầu tiên của DataFrame dưới dạng bảng để chúng ta dễ dàng quan sát cấu trúc dữ liệu.

**Kết quả mong đợi:** Một bảng gồm 5 dòng với các cột như `user_id`, `event_type`, `timestamp`, `session_duration`, `device`, v.v. Các giá trị sẽ hiển thị dưới dạng chuỗi hoặc số tùy thuộc vào dữ liệu gốc.

In [ ]:
events_df.printSchema()

**Giải thích cú pháp:**

- `.printSchema()`: In ra cấu trúc (schema) của DataFrame dưới dạng cây thư mục, bao gồm tên cột và kiểu dữ liệu (ví dụ: `string`, `long`, `double`).

**Kết quả mong đợi:** Danh sách các cột và kiểu dữ liệu tương ứng. Lưu ý rằng cột `timestamp` ban đầu thường có kiểu `string` vì nó được đọc trực tiếp từ JSON. Chúng ta sẽ cần chuyển đổi nó ở bước tiếp theo.

In [ ]:
processed_events = events_df \
    .withColumn("timestamp", to_timestamp(col("timestamp"))) \
    .withColumn("hour", hour(col("timestamp"))) \
    .withColumn("session_length_minutes", round(col("session_duration") / 60, 2)) \
    .withColumn("is_conversion", col("event_type") == "purchase")

processed_events.show(5)

**Giải thích cú pháp:**

- `.withColumn(ten_cot, bieu_thuc)`: Thêm một cột mới hoặc ghi đè cột cũ trong DataFrame.
- `to_timestamp(col("timestamp"))`: Chuyển đổi cột `timestamp` từ kiểu chuỗi (`string`) sang kiểu thời gian (`timestamp`) để có thể tính toán.
- `hour(col("timestamp"))`: Trích xuất phần giờ (từ 0 đến 23) từ cột timestamp vừa được chuyển đổi.
- `round(col("session_duration") / 60, 2)`: Chia thời lượng phiên cho 60 để đổi từ giây sang phút, sau đó làm tròn đến 2 chữ số thập phân.
- `col("event_type") == "purchase"`: So sánh giá trị cột `event_type` với chuỗi `"purchase"`. Trả về `true` nếu khớp, `false` nếu không. Kết quả là một cột kiểu boolean.

**Kết quả mong đợi:** Bảng 5 dòng giống bước trước nhưng có thêm 3 cột mới: `hour` (số nguyên), `session_length_minutes` (số thực), và `is_conversion` (true/false). Cột `timestamp` bây giờ có định dạng chuẩn của kiểu timestamp thay vì chuỗi thô.

In [ ]:
hourly_stats = processed_events.groupBy("hour", "event_type").count()

hourly_stats_ordered = hourly_stats.orderBy("hour", "event_type")

hourly_stats_ordered.show()

**Giải thích cú pháp:**

- `.groupBy("hour", "event_type")`: Gom nhóm dữ liệu theo hai cột là `hour` và `event_type`. Mọi tổ hợp giờ-loại sự kiện duy nhất sẽ tạo thành một nhóm.
- `.count()`: Đếm số lượng bản ghi trong mỗi nhóm vừa gom.
- `.orderBy("hour", "event_type")`: Sắp xếp kết quả tăng dần theo cột `hour` trước, sau đó đến `event_type` để dễ đọc hơn.

**Kết quả mong đợi:** Một bảng gồm 3 cột: `hour`, `event_type`, và `count`. Mỗi dòng thể hiện số lần một loại sự kiện xảy ra trong một giờ cụ thể. Bảng được sắp xếp thứ tự từ giờ 0 đến 23.

In [ ]:
user_sessions = processed_events.groupBy("user_id").agg(
    count("*").alias("total_events"),
    max("session_duration").alias("max_session_duration"),
    countDistinct("device").alias("devices_used"),
    sum(col("is_conversion").cast("int")).alias("conversions")
)

user_sessions.show()

**Giải thích cú pháp:**

- `.groupBy("user_id")`: Gom nhóm toàn bộ dữ liệu theo từng người dùng (`user_id`).
- `.agg(...)`: Áp dụng nhiều hàm tổng hợp cùng lúc trên mỗi nhóm.
- `count("*").alias("total_events")`: Đếm tổng số sự kiện của người dùng đó. `.alias()` dùng để đặt tên lại cho cột.
- `max("session_duration").alias("max_session_duration")`: Tìm thời lượng phiên lớn nhất của người dùng.
- `countDistinct("device").alias("devices_used")`: Đếm số lượng thiết bị khác nhau mà người dùng đã sử dụng.
- `sum(col("is_conversion").cast("int"))`: Vì `is_conversion` là kiểu boolean (true/false), ta dùng `.cast("int")` để chuyển true thành 1 và false thành 0, sau đó dùng `sum` để đếm tổng số lần mua hàng.

**Kết quả mong đợi:** Bảng gồm 5 cột: `user_id`, `total_events`, `max_session_duration`, `devices_used`, `conversions`. Mỗi dòng đại diện cho một người dùng và các chỉ số thống kê tương ứng.

In [ ]:
high_activity_users = user_sessions.filter(col("total_events") > 50)

print("Số lượng người dùng có hoạt động cao (trên 50 sự kiện):", high_activity_users.count())

high_activity_users.show()

**Giải thích cú pháp:**

- `.filter(col("total_events") > 50)`: Lọc DataFrame `user_sessions`, chỉ giữ lại những dòng mà cột `total_events` có giá trị lớn hơn 50. Đây là cách mô phỏng việc đưa ra cảnh báo (alert) cho những người dùng hoạt động bất thường nhiều.
- `.count()`: Đếm số lượng dòng còn lại sau khi lọc (tức là số người dùng thỏa mãn điều kiện).
- `.show()`: Hiển thị danh sách chi tiết những người dùng này.

**Kết quả mong đợi:** Dòng chữ in ra màn hình cho biết có bao nhiêu người dùng có trên 50 sự kiện. Tiếp theo là một bảng (có thể trống nếu không có ai đạt ngưỡng) liệt kê thông tin của những người dùng đó.

In [ ]:
total_events = processed_events.count()
unique_users = processed_events.select("user_id").distinct().count()
total_conversions = processed_events.filter(col("is_conversion") == True).count()

conversion_rate = (total_conversions / total_events) * 100 if total_events > 0 else 0

print("--- Thống kê tổng quan ---")
print("Tổng số sự kiện:", total_events)
print("Số người dùng duy nhất:", unique_users)
print("Tỷ lệ chuyển đổi (%):", round(conversion_rate, 2))

**Giải thích cú pháp:**

- `processed_events.count()`: Đếm tổng số dòng (sự kiện) trong toàn bộ DataFrame.
- `.select("user_id").distinct().count()`: Chỉ lấy cột `user_id`, loại bỏ các giá trị trùng lặp bằng `.distinct()`, rồi đếm để ra số người dùng duy nhất.
- `.filter(col("is_conversion") == True).count()`: Lọc ra các sự kiện là giao dịch mua hàng và đếm số lượng.
- `(total_conversions / total_events) * 100`: Công thức tính tỷ lệ chuyển đổi phần trăm. Phần `if total_events > 0 else 0` là kiểm tra điều kiện để tránh lỗi chia cho 0 nếu dữ liệu trống.
- `round(conversion_rate, 2)`: Làm tròn tỷ lệ chuyển đổi đến 2 chữ số thập phân khi in ra.

**Kết quả mong đợi:** Ba dòng in ra màn hình hiển thị tổng số sự kiện, số người dùng duy nhất, và tỷ lệ chuyển đổi tính bằng phần trăm.

## Tổng kết

Trong bài thực hành này, chúng ta đã hoàn thành thử thách mô phỏng luồng xử lý streaming bằng batch:

1. **Đọc và khám phá dữ liệu**: Sử dụng `spark.read.json` để nạp dữ liệu sự kiện web và kiểm tra schema.
2. **Xử lý và biến đổi dữ liệu**: Dùng `withColumn` để thêm các cột phái sinh như trích xuất giờ, chuyển đổi đơn vị thời gian, và tạo cờ boolean.
3. **Thống kê theo nhóm**: Áp dụng `groupBy` kết hợp với `count`, `agg`, `max`, `countDistinct`, `sum` để phân tích hành vi người dùng và tần suất sự kiện theo giờ.
4. **Lọc dữ liệu (Alerting)**: Dùng `filter` để tìm ra những người dùng có hoạt động bất thường dựa trên một ngưỡng cho trước.
5. **Tính toán chỉ số tổng quan**: Tổng hợp các con số kinh doanh cơ bản như tỷ lệ chuyển đổi.

Mặc dù đây không phải là Structured Streaming thực sự (dữ liệu không đến liên tục), cách tiếp cận xử lý theo batch này giúp hiểu rõ các phép biến đổi cốt lõi trước khi áp dụng vào môi trường streaming thực tế.